In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import xgboost as xgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

### Loading data

In [ ]:
df = pd.read_csv("../data/processed/features.csv")
print(f"Shape: {df.shape}")
print(f"Train rows: {(~df['is_test']).sum()}")
print(f"Test rows:  {df['is_test'].sum()}")

### Train validation split

In [ ]:
feature_cols = [
    "cumulative_energy_MJ", "time_since_heating_s",
    *[f"Bulk_{i}_cum" for i in range(1, 16)],
    "total_bulk_cum",
    *[f"Wire_{i}_cum" for i in range(1, 10)],
    "total_wire_cum",
    "gas_total", "prev_temp", "measurement_index",
]

train_df = df[~df["is_test"]].copy()
test_df = df[df["is_test"]].copy()

X_train = train_df[feature_cols]
y_train = train_df["Temperature"]
groups = train_df["key"]

X_test = test_df[feature_cols]
y_test = test_df["true_temp"]

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Features: {len(feature_cols)}")
print(f"Unique melts in train: {groups.nunique()}")

In [ ]:
first_measurements = df[df["measurement_index"] == 1]
first_in_test = first_measurements[first_measurements["is_test"] == True]
first_in_train = first_measurements[first_measurements["is_test"] == False]

print(f"Total melts: {df['key'].nunique()}")
print(f"First measurements in train: {len(first_in_train)}")
print(f"First measurements in test:  {len(first_in_test)}")
print(f"\nMelts where first measurement is in test: {len(first_in_test)}")
print(f"As % of total melts: {len(first_in_test)/df['key'].nunique()*100:.1f}%")


  Why GroupKFold by heat

 Random row-level splitting would be invalid here for two reasons:

 1. **Sequential correlation** — consecutive measurements within a heat are
    not independent. A model trained on measurement 3 of heat X and asked to
    predict measurement 4 of the same heat has seen nearly identical feature
    values and a target that is typically within 10–20°C of its prediction
    target. This inflates validation metrics beyond what the model can deliver
    on truly unseen heats.

 2. **prev_temp leakage** — the previous temperature feature directly
    encodes a target value from the same heat. If measurements from the same
    heat appear in both train and validation, the model effectively sees
    near-copies of validation targets during training.

 GroupKFold ensures that all measurements from a given heat are in the same
 fold — the model must generalize across melts, not within them.

In [ ]:
gkf = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    train_keys = groups.iloc[train_idx].nunique()
    val_keys = groups.iloc[val_idx].nunique()
    print(f"Fold {fold + 1}: train {len(train_idx)} rows ({train_keys} melts) | "
          f"val {len(val_idx)} rows ({val_keys} melts)")

 
 ### Split structure

 The 5-fold GroupKFold produces balanced folds with approximately 643 melts
 (2600 rows) in validation and 2572 melts (10400 rows) in training per fold.
 The total of 3215 unique melts in the training set confirms that the 2 melts
 missing from the arc data (identified in Notebook 01) propagated correctly
 through the feature matrix — they are present with zero energy features but
 valid temperature targets.

 The group constraint guarantees that no melt appears in both train and
 validation within any fold. This means the model is always evaluated on
 melts it has never seen — the same condition it will face on the held-out
 test set.

### Baseline model

In [ ]:
global_mean = y_train.mean()
print(f"Global mean temperature: {global_mean:.1f} °C")

In [ ]:
baseline_maes = []
baseline_rmses = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    val_X = X_train.iloc[val_idx]
    val_y = y_train.iloc[val_idx]

    # Predict prev_temp where available, global mean otherwise
    baseline_pred = val_X["prev_temp"].fillna(global_mean)

    mae = mean_absolute_error(val_y, baseline_pred)
    rmse = root_mean_squared_error(val_y, baseline_pred)
    baseline_maes.append(mae)
    baseline_rmses.append(rmse)
    print(f"Fold {fold + 1}: MAE = {mae:.2f} °C  |  RMSE = {rmse:.2f} °C")

print(f"\nBaseline mean MAE:  {np.mean(baseline_maes):.2f} ± {np.std(baseline_maes):.2f} °C")
print(f"Baseline mean RMSE: {np.mean(baseline_rmses):.2f} ± {np.std(baseline_rmses):.2f} °C")


 ### Baseline performance

 The "no change" baseline achieves a mean MAE (Mean Absolute Error) of 11.36 ± 0.26°C and RMSE (Root Mean Squared Error) of
 16.30 ± 0.46°C across 5 folds. The low variance across folds confirms that
 GroupKFold is producing representative splits — no fold contains a
 disproportionate share of easy or difficult melts.

 The global mean temperature of 1592°C reflects the typical LF operating window:
 steel arrives from primary refining at roughly 1550–1570°C and is heated to a
 tapping target of 1600–1620°C, depending on the grade and caster requirements.

 An 11°C average error from this trivial predictor is operationally significant.
 LF tapping temperature windows are typically ±15–20°C — meaning the baseline
 already accounts for roughly half the allowable deviation using nothing more
 than the previous reading. This sets a meaningful bar: XGBoost must learn the
 cumulative effects of heating, additions, and time to improve substantially
 beyond what "assume nothing changed" already provides.

 The RMSE (16.3°C) exceeds the MAE by ~44%, indicating the presence of large
 residuals — likely concentrated on first measurements (where prev_temp is
 unavailable and the baseline falls back to the global mean) and on measurements
 following large thermal events (bulk additions, extended heating) where the
 "no change" assumption breaks down most severely.

### XGBoost - default parameters

In [ ]:
default_params = {
    "n_estimators": 500,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "n_jobs": -1,
}

print("Default parameters:")
for k, v in default_params.items():
    print(f"  {k}: {v}")

In [ ]:
default_maes = []
default_rmses = []
default_models = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    fold_X_train = X_train.iloc[train_idx]
    fold_y_train = y_train.iloc[train_idx]
    fold_X_val = X_train.iloc[val_idx]
    fold_y_val = y_train.iloc[val_idx]

    model = xgb.XGBRegressor(**default_params)
    model.fit(
        fold_X_train, fold_y_train,
        eval_set=[(fold_X_val, fold_y_val)],
        verbose=False,
    )

    preds = model.predict(fold_X_val)
    mae = mean_absolute_error(fold_y_val, preds)
    rmse = root_mean_squared_error(fold_y_val, preds)
    default_maes.append(mae)
    default_rmses.append(rmse)
    default_models.append(model)

    print(f"Fold {fold + 1}: MAE = {mae:.2f} °C  |  RMSE = {rmse:.2f} °C")

print(f"\nDefault XGBoost mean MAE:  {np.mean(default_maes):.2f} ± {np.std(default_maes):.2f} °C")
print(f"Default XGBoost mean RMSE: {np.mean(default_rmses):.2f} ± {np.std(default_rmses):.2f} °C")

In [ ]:
improvement_mae = np.mean(baseline_maes) - np.mean(default_maes)
improvement_pct = improvement_mae / np.mean(baseline_maes) * 100
print(f"MAE improvement over baseline: {improvement_mae:.2f} °C ({improvement_pct:.1f}%)")


 ### Default XGBoost performance

 The default XGBoost model achieves a mean MAE of 9.81 ± 0.26°C and RMSE of
 15.09 ± 0.46°C — a 1.56°C (13.7%) improvement in MAE over the baseline.

 The improvement is modest in absolute terms, but this is expected: the baseline
 was physically strong. Predicting "temperature hasn't changed since the last
 reading" already captures the dominant signal — the incremental nature of the
 LF process. What XGBoost adds is the ability to correct for the perturbations
 that make temperature change between consecutive readings: energy delivered
 by heating sessions, thermal losses from additions and elapsed time, and
 process stage effects.

 The larger relative reduction in RMSE (1.21°C, 7.4%) compared to MAE indicates
 the model is particularly effective at correcting large errors — cases where the
 baseline's "no change" assumption breaks down most severely. These are
 operationally the most important predictions: a 30°C error on a measurement
 following a large bulk addition could lead to incorrect heating decisions,
 while a 5°C error on a stable measurement has little operational consequence.

 Fold variance remains low and mirrors the baseline pattern, confirming that
 the GroupKFold splits are representative and the model is not overfitting to
 any particular subset of melts.

### Hyperparameter tuning

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
    }

    fold_maes = []
    for train_idx, val_idx in gkf.split(X_train, y_train, groups):
        fold_X_train = X_train.iloc[train_idx]
        fold_y_train = y_train.iloc[train_idx]
        fold_X_val = X_train.iloc[val_idx]
        fold_y_val = y_train.iloc[val_idx]

        model = xgb.XGBRegressor(**params)
        model.fit(
            fold_X_train, fold_y_train,
            eval_set=[(fold_X_val, fold_y_val)],
            verbose=False,
        )

        preds = model.predict(fold_X_val)
        fold_maes.append(mean_absolute_error(fold_y_val, preds))

    return np.mean(fold_maes)

In [ ]:
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"Best MAE: {study.best_value:.2f} °C")
print(f"\nBest parameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
best_params = study.best_params
best_params["random_state"] = 42
best_params["n_jobs"] = -1

tuned_maes = []
tuned_rmses = []
tuned_models = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    fold_X_train = X_train.iloc[train_idx]
    fold_y_train = y_train.iloc[train_idx]
    fold_X_val = X_train.iloc[val_idx]
    fold_y_val = y_train.iloc[val_idx]

    model = xgb.XGBRegressor(**best_params)
    model.fit(
        fold_X_train, fold_y_train,
        eval_set=[(fold_X_val, fold_y_val)],
        verbose=False,
    )

    preds = model.predict(fold_X_val)
    mae = mean_absolute_error(fold_y_val, preds)
    rmse = root_mean_squared_error(fold_y_val, preds)
    tuned_maes.append(mae)
    tuned_rmses.append(rmse)
    tuned_models.append(model)

    print(f"Fold {fold + 1}: MAE = {mae:.2f} °C  |  RMSE = {rmse:.2f} °C")

print(f"\nTuned XGBoost mean MAE:  {np.mean(tuned_maes):.2f} ± {np.std(tuned_maes):.2f} °C")
print(f"Tuned XGBoost mean RMSE: {np.mean(tuned_rmses):.2f} ± {np.std(tuned_rmses):.2f} °C")

In [ ]:
print("Performance comparison:")
print(f"  Baseline:        MAE = {np.mean(baseline_maes):.2f} °C  |  RMSE = {np.mean(baseline_rmses):.2f} °C")
print(f"  Default XGBoost: MAE = {np.mean(default_maes):.2f} °C  |  RMSE = {np.mean(default_rmses):.2f} °C")
print(f"  Tuned XGBoost:   MAE = {np.mean(tuned_maes):.2f} °C  |  RMSE = {np.mean(tuned_rmses):.2f} °C")

improvement_vs_baseline = np.mean(baseline_maes) - np.mean(tuned_maes)
improvement_vs_default = np.mean(default_maes) - np.mean(tuned_maes)
print(f"\n  Improvement vs baseline: {improvement_vs_baseline:.2f} °C ({improvement_vs_baseline/np.mean(baseline_maes)*100:.1f}%)")
print(f"  Improvement vs default:  {improvement_vs_default:.2f} °C ({improvement_vs_default/np.mean(default_maes)*100:.1f}%)")

### Hyperparameter tuning results

 The tuned model achieves a mean MAE of 9.08 ± 0.22°C and RMSE of
 13.91 ± 0.48°C — a 20.0% improvement over the baseline and 7.4% over the
 default XGBoost configuration.

 | Model | MAE (°C) | RMSE (°C) |
 |---|---|---|
 | Baseline (prev_temp) | 11.36 ± 0.26 | 16.30 ± 0.46 |
 | XGBoost default | 9.81 ± 0.26 | 15.09 ± 0.46 |
 | XGBoost tuned | 9.08 ± 0.22 | 13.91 ± 0.48 |

 The tuned parameters reveal how the model adapts to this dataset:

 - **Low learning rate (0.013) with ~500 trees** — the model builds its
   predictions gradually through many small corrections rather than few large
   ones. This is appropriate for a target with low variance (most temperatures
   fall within a 100°C window) where aggressive fitting would chase noise.

 - **Strong L1 regularization (reg_alpha = 1.60)** — L1 promotes sparsity by
   driving uninformative feature weights toward zero. With 15 individual bulk
   and 9 individual wire columns that are mostly zero, this is the model's
   mechanism for effectively ignoring rare materials without manual feature
   selection. The SHAP analysis in Notebook 04 will reveal which materials
   survive this regularization.

 - **Weak L2 regularization (reg_lambda = 0.004)** — L2 penalizes large weights
   uniformly, and the near-zero value indicates the model does not need to
   dampen any dominant feature. This makes sense: `prev_temp` is expected to
   dominate naturally, and suppressing it would hurt performance.

 - **max_depth = 5, min_child_weight = 4** — moderate tree depth with a minimum
   leaf size constraint prevents overfitting to small subgroups of melts while
   allowing enough complexity to capture interactions (e.g., the effect of bulk
   additions depends on how much energy has been delivered).

 The RMSE improvement from default to tuned (15.09 → 13.91, −7.8%) is larger
 than the MAE improvement (9.81 → 9.08, −7.4%), confirming that tuning
 primarily helps on the hardest predictions — the large-residual cases where
 the default model failed to capture complex thermal dynamics.

### Residual analysis

Collecting residuals from all validation folds

In [ ]:
all_val_indices = []
all_residuals = []
all_preds = []
all_actuals = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    fold_X_val = X_train.iloc[val_idx]
    fold_y_val = y_train.iloc[val_idx]

    model = tuned_models[fold]
    preds = model.predict(fold_X_val)
    residuals = fold_y_val.values - preds

    all_val_indices.extend(val_idx)
    all_residuals.extend(residuals)
    all_preds.extend(preds)
    all_actuals.extend(fold_y_val.values)

residual_df = pd.DataFrame({
    "actual": all_actuals,
    "predicted": all_preds,
    "residual": all_residuals,
    "measurement_index": X_train.iloc[all_val_indices]["measurement_index"].values,
    "prev_temp": X_train.iloc[all_val_indices]["prev_temp"].values,
    "cumulative_energy_MJ": X_train.iloc[all_val_indices]["cumulative_energy_MJ"].values,
    "total_bulk_cum": X_train.iloc[all_val_indices]["total_bulk_cum"].values,
})

print(f"Residual stats:")
print(f"  Mean:   {residual_df['residual'].mean():.2f} °C")
print(f"  Std:    {residual_df['residual'].std():.2f} °C")
print(f"  Median: {residual_df['residual'].median():.2f} °C")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residuals vs predicted
axes[0].scatter(residual_df["predicted"], residual_df["residual"], alpha=0.15, s=10)
axes[0].axhline(0, color="red", linewidth=1)
axes[0].set_xlabel("Predicted Temperature (°C)")
axes[0].set_ylabel("Residual (°C)")
axes[0].set_title("Residuals vs Predicted")

# Residual distribution
residual_df["residual"].hist(bins=60, ax=axes[1], edgecolor="black", alpha=0.7)
axes[1].axvline(0, color="red", linewidth=1)
axes[1].set_xlabel("Residual (°C)")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution")

# Actual vs predicted
axes[2].scatter(residual_df["actual"], residual_df["predicted"], alpha=0.15, s=10)
lims = [residual_df["actual"].min() - 10, residual_df["actual"].max() + 10]
axes[2].plot(lims, lims, color="red", linewidth=1)
axes[2].set_xlabel("Actual Temperature (°C)")
axes[2].set_ylabel("Predicted Temperature (°C)")
axes[2].set_title("Actual vs Predicted")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

residual_by_idx = residual_df.groupby("measurement_index")["residual"].agg(["mean", "std", "count"])
residual_by_idx = residual_by_idx[residual_by_idx["count"] >= 20]

axes[0].bar(residual_by_idx.index, residual_by_idx["mean"], 
            yerr=residual_by_idx["std"], alpha=0.7, capsize=3, edgecolor="black")
axes[0].axhline(0, color="red", linewidth=1)
axes[0].set_xlabel("Measurement Index")
axes[0].set_ylabel("Mean Residual (°C)")
axes[0].set_title("Residuals by Measurement Position")

# MAE by measurement index
mae_by_idx = residual_df.groupby("measurement_index")["residual"].apply(
    lambda x: np.mean(np.abs(x))
)
mae_by_idx = mae_by_idx[residual_by_idx.index]

axes[1].bar(mae_by_idx.index, mae_by_idx.values, alpha=0.7, edgecolor="black")
axes[1].set_xlabel("Measurement Index")
axes[1].set_ylabel("MAE (°C)")
axes[1].set_title("MAE by Measurement Position")

plt.tight_layout()
plt.show()

Residuals by prev_temp availability

In [ ]:
has_prev = residual_df["prev_temp"].notna()
mae_with_prev = np.mean(np.abs(residual_df.loc[has_prev, "residual"]))
mae_without_prev = np.mean(np.abs(residual_df.loc[~has_prev, "residual"]))

print(f"MAE with prev_temp available:    {mae_with_prev:.2f} °C  ({has_prev.sum()} rows)")
print(f"MAE without prev_temp (1st meas): {mae_without_prev:.2f} °C  ({(~has_prev).sum()} rows)")

 ### Residual patterns

 **Global bias** — the mean residual is effectively zero (−0.00°C) with a median
 of −0.15°C, confirming the model has no systematic bias. The residual
 distribution is approximately symmetric around zero with a standard deviation
 of 13.92°C, consistent with the cross-validated RMSE of 13.91°C.

 **Heteroscedasticity** — the residuals-vs-predicted plot shows higher variance
 in the 1580–1600°C range, where the majority of heats operate, and lower
 variance at the extremes. This is expected: the dense operating region contains
 the widest diversity of process conditions — different grades, addition loads,
 and heating histories all converge to similar temperatures but through different
 paths. At the extremes, the process conditions are more constrained (very cold
 arrivals or intentional superheating), making prediction easier.

 The actual-vs-predicted plot shows mild regression toward the mean: the model
 underestimates high temperatures and overestimates low ones. This is a known
 behavior of tree-based models — predictions are bounded by the range of training
 targets seen in each leaf, so extreme values are pulled inward. For this
 application, the effect is minor and does not warrant correction.

 **Measurement index effect** — MAE drops sharply from ~19°C at the first
 measurement to ~5°C by the third and subsequent measurements. This is the
 single most important diagnostic in this analysis. The first measurement in
 each heat has no `prev_temp` — the model must estimate arrival temperature
 from cumulative energy, additions, and gas alone, without the anchor of a
 prior thermal state. This is a fundamentally harder problem: arrival
 temperature depends on upstream variables not in this dataset — tapping
 temperature from the primary furnace, ladle transit time, ladle refractory
 condition, and waiting time before LF treatment begins.

 **prev_temp availability** — the split quantifies the effect precisely:
 MAE = 5.86°C when `prev_temp` is available (9784 rows) vs. 18.90°C without
 it (3215 rows). The feature is transformative, not merely important — it
 shifts the model from near-baseline performance to operationally useful
 precision. This is physically intuitive: the LF process is incremental, and
 knowing the current thermal state reduces the prediction problem from "estimate
 absolute temperature from process inputs" to "estimate the temperature change
 caused by recent events."

 **Operational implication** — this model is most valuable for intermediate
 measurements within a heat, where `prev_temp` provides a strong anchor. For
 arrival temperature estimation (first measurement, no prior reading), the
 model performs at near-baseline level. Improving first-measurement predictions
 would require features not available in this dataset — ladle campaign sequence
 number, time since last use, upstream tapping temperature — or a dedicated
 sub-model trained specifically on arrival conditions.

### Final model - test set evaluation

In [ ]:
final_model = xgb.XGBRegressor(**best_params)
final_model.fit(X_train, y_train, verbose=False)

test_preds = final_model.predict(X_test)
test_mae = mean_absolute_error(y_test, test_preds)
test_rmse = root_mean_squared_error(y_test, test_preds)

print(f"Test set MAE:  {test_mae:.2f} °C")
print(f"Test set RMSE: {test_rmse:.2f} °C")

print(f"\nComparison:")
print(f"  CV MAE:   {np.mean(tuned_maes):.2f} ± {np.std(tuned_maes):.2f} °C")
print(f"  Test MAE: {test_mae:.2f} °C")

Test set residual analysis

In [ ]:
test_residuals = y_test.values - test_preds

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(test_preds, test_residuals, alpha=0.2, s=10)
axes[0].axhline(0, color="red", linewidth=1)
axes[0].set_xlabel("Predicted Temperature (°C)")
axes[0].set_ylabel("Residual (°C)")
axes[0].set_title("Test Set — Residuals vs Predicted")

(pd.Series(test_residuals)).hist(bins=50, ax=axes[1], edgecolor="black", alpha=0.7)
axes[1].axvline(0, color="red", linewidth=1)
axes[1].set_xlabel("Residual (°C)")
axes[1].set_ylabel("Count")
axes[1].set_title("Test Set — Residual Distribution")

axes[2].scatter(y_test, test_preds, alpha=0.2, s=10)
lims = [y_test.min() - 10, y_test.max() + 10]
axes[2].plot(lims, lims, color="red", linewidth=1)
axes[2].set_xlabel("Actual Temperature (°C)")
axes[2].set_ylabel("Predicted Temperature (°C)")
axes[2].set_title("Test Set — Actual vs Predicted")

plt.tight_layout()
plt.show()

MAE breakdown by prev_temp availability on test set

In [ ]:
test_has_prev = X_test["prev_temp"].notna()
test_mae_with = mean_absolute_error(y_test[test_has_prev], test_preds[test_has_prev])
test_mae_without = mean_absolute_error(y_test[~test_has_prev], test_preds[~test_has_prev])

print(f"Test MAE with prev_temp:    {test_mae_with:.2f} °C  ({test_has_prev.sum()} rows)")
print(f"Test MAE without prev_temp: {test_mae_without:.2f} °C  ({(~test_has_prev).sum()} rows)")

Saving predictions

In [ ]:
test_output = test_df[["key", "time", "true_temp"]].copy()
test_output["predicted_temp"] = test_preds
test_output["residual"] = test_output["true_temp"] - test_output["predicted_temp"]
test_output.to_csv("../data/processed/test_predictions.csv", index=False)
print(f"Saved to data/processed/test_predictions.csv — {len(test_output)} rows")

 ### Test set results

 The test set MAE of 15.53°C is substantially higher than the cross-validated
 estimate of 9.08°C. This gap is not caused by overfitting — it is caused by
 a structural difference in the composition of the test set.

 The diagnostic is in the `prev_temp` split:

 | Subset | CV validation | Test set |
 |---|---|---|
 | With prev_temp | MAE 5.86°C (76% of rows) | MAE 6.78°C (25% of rows) |
 | Without prev_temp | MAE 18.90°C (24% of rows) | MAE 18.52°C (75% of rows) |

 The test set is dominated by rows without a previous temperature anchor — 2162
 of 2900 rows (75%) have `prev_temp = NaN`. In cross-validation, this proportion
 was inverted: only 24% of validation rows lacked `prev_temp`. The test rows were
 selected by the original dataset creators in a way that disproportionately masks
 consecutive measurements, causing NaN to propagate forward through the `prev_temp`
 chain.

 Within each regime, the model generalizes correctly:

 - With `prev_temp` available, test MAE (6.78°C) is actually better than CV
   (5.86°C), because the final model was trained on the full 12999 rows rather
   than 80%. The model has learned the thermal dynamics well.
 - Without `prev_temp`, test MAE (18.52°C) is consistent with CV (18.90°C). The
   model performs the same on unseen first-measurement-like conditions as it did
   during validation.

 The aggregate test MAE is higher simply because the test set samples more heavily
 from the harder regime. This is a property of the evaluation set, not a failure
 of the model.

 The residual plots confirm the same patterns seen in cross-validation:
 heteroscedasticity concentrated in the 1575–1600°C operating range, a positive
 skew in the residual distribution reflecting the difficulty of predicting
 without `prev_temp`, and regression toward the mean in the actual-vs-predicted
 plot — now more pronounced because the model defaults toward the global mean
 when the previous temperature anchor is absent.

 The actual-vs-predicted plot makes this visually stark: predictions cluster in
 a narrow horizontal band around 1575–1590°C for the majority of test rows,
 while actual temperatures span the full 1520–1700°C range. This is the model
 predicting the mean thermal state when it lacks the information to predict
 deviations from it — a physically honest behavior.

 ## Summary

 This notebook trained an XGBoost regressor to predict steel temperature at each
 measurement point within a Ladle Furnace melt, using physically motivated
 features with strict temporal causality.

 **Key results:**

 | Model | CV MAE (°C) | Test MAE (°C) |
 |---|---|---|
 | Baseline (prev_temp) | 11.36 ± 0.26 | — |
 | XGBoost default | 9.81 ± 0.26 | — |
 | XGBoost tuned | 9.08 ± 0.22 | 15.53 |

 **Key findings:**

 1. **Previous temperature dominates prediction quality.** With `prev_temp`
    available, the model achieves ~6°C MAE — within the noise floor of industrial
    temperature measurement (thermocouple uncertainty is typically ±3–5°C). Without
    it, MAE rises to ~18°C, near baseline level.

 2. **The test set is structurally harder than cross-validation.** 75% of test
    rows lack `prev_temp`, versus 24% in CV validation folds. The model generalizes
    correctly within each regime — the aggregate gap reflects composition, not
    overfitting.

 3. **The tuned model learns meaningful thermal dynamics.** The improvement over
    baseline (20% MAE reduction in CV) comes from integrating cumulative energy
    input, time-based heat loss, and material addition effects — physical mechanisms
    that the simple "no change" baseline ignores.

 4. **First-measurement prediction is the primary limitation.** Without a prior
    temperature anchor, the model lacks information about upstream conditions
    (tapping temperature, ladle thermal history) that determine arrival temperature.
    This is a data limitation, not a model limitation.

 Notebook 04 will use SHAP to explain which features drive individual predictions
 and connect the model's learned relationships back to LF thermodynamics and
 refining kinetics.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
})

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("#F8F8F8")
ax.set_facecolor("#F8F8F8")

ax.scatter(
    residual_df["actual"],
    residual_df["predicted"],
    alpha=0.12,
    s=8,
    color="#4C72B0",
)

lims = [
    min(residual_df["actual"].min(), residual_df["predicted"].min()) - 10,
    max(residual_df["actual"].max(), residual_df["predicted"].max()) + 10,
]
ax.plot(lims, lims, color="#C44E52", linewidth=1.2, label="Perfect prediction")
ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel("Actual temperature (°C)", fontsize=11)
ax.set_ylabel("Predicted temperature (°C)", fontsize=11)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", labelsize=10)

ax.legend(fontsize=10, framealpha=0.5)

# Annotate with key metrics
mae_with = 5.86
mae_without = 18.90
annotation = (
    f"MAE (with prior temperature): {mae_with:.2f} °C\n"
    f"MAE (first measurement only): {mae_without:.2f} °C"
)
ax.text(
    0.03, 0.97, annotation,
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="top",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.7, edgecolor="lightgray"),
)

plt.title(
    "Predicted vs actual steel temperature — Ladle Furnace model",
    fontsize=13,
    fontweight="bold",
    pad=14,
    loc="left",
)

plt.tight_layout(pad=1.5)
plt.savefig("actual_vs_predicted_linkedin.png", dpi=180, bbox_inches="tight", facecolor="#F8F8F8")
plt.show()
print("Saved to actual_vs_predicted_linkedin.png")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("#F8F8F8")
ax.set_facecolor("#F8F8F8")

has_prev = residual_df["prev_temp"].notna()

ax.scatter(
    residual_df.loc[has_prev, "actual"],
    residual_df.loc[has_prev, "predicted"],
    alpha=0.12, s=8, color="#4C72B0",
    label=f"With prior temperature — MAE: 5.86 °C"
)

ax.scatter(
    residual_df.loc[~has_prev, "actual"],
    residual_df.loc[~has_prev, "predicted"],
    alpha=0.25, s=8, color="#C44E52",
    label=f"First measurement only — MAE: 18.90 °C"
)

lims = [
    min(residual_df["actual"].min(), residual_df["predicted"].min()) - 10,
    max(residual_df["actual"].max(), residual_df["predicted"].max()) + 10,
]
ax.plot(lims, lims, color="black", linewidth=1.0, linestyle="--", alpha=0.5)
ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel("Actual temperature (°C)", fontsize=11)
ax.set_ylabel("Predicted temperature (°C)", fontsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", labelsize=10)
ax.legend(fontsize=10, framealpha=0.6, loc="upper left")

plt.title(
    "Predicted vs actual steel temperature — Ladle Furnace model",
    fontsize=13, fontweight="bold", pad=14, loc="left",
)

plt.tight_layout(pad=1.5)
plt.savefig("actual_vs_predicted_linkedin.png", dpi=180, bbox_inches="tight", facecolor="#F8F8F8")
plt.show()